In [ ]:
""" Author: Levi McRea
    Purpose: To pull API data from EIA and Open-Mateo API's to build visuals and correlations based on weather patterns and energy consumption
    Date Last Edited: 4/6/2026
"""

#installation of libraries
!pip install sqlalchemy psycopg2-binary
!pip install azure.storage.blob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 35.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 15.5 MB/s eta 0:00:00


In [ ]:
# API call & dataframe creation for EIA's Energy Consumption API

import requests
import pandas as pd
from google.colab import userdata

# Pull your key from Colab secrets — never hardcode this
EIA_KEY = userdata.get("EIA_KEY")

def get_energy(state_code, start, end):
    """
    Pulls monthly residential electricity consumption from EIA API

    Parameters:
        state_code : str  — two letter state code e.g. "TX", "MN", "CA"
        start      : str  — start month in YYYY-MM format e.g. "2024-01"
        end        : str  — end month in YYYY-MM format e.g. "2024-12"

    Returns:
        Pandas dataframe with columns:
        period, state, consumption_million_kwh
    """
    url = "https://api.eia.gov/v2/electricity/retail-sales/data/"

    params = {
        "api_key": EIA_KEY,
        "frequency": "monthly",        # monthly aligns with weather aggregation
        "data[0]": "sales",            # electricity consumed in million kWh
        "facets[stateid][]": state_code,
        "facets[sectorid][]": "RES",   # residential — strongest weather signal
        "start": start,
        "end": end,
        "sort[0][column]": "period",   # sort by date
        "sort[0][direction]": "asc"    # oldest to newest
    }

    response = requests.get(url, params=params)

    # Check the call succeeded before trying to parse
    if response.status_code != 200:
        print(f"Error {response.status_code} for {state_code}: {response.text}")
        return pd.DataFrame()

    # Parse response into dataframe
    df = pd.DataFrame(response.json()["response"]["data"])

    # Keep only what you need and rename for clarity
    df = df[["period", "stateid", "sales"]].rename(columns={
        "stateid": "state",
        "sales": "consumption_million_kwh"
    })

    # Ensure numeric — API sometimes returns strings
    df["consumption_million_kwh"] = pd.to_numeric(
        df["consumption_million_kwh"], errors="coerce"
    )

    # Add a billion kWh column for cleaner chart labels
    df["consumption_billion_kwh"] = df["consumption_million_kwh"] / 1000

    return df


# All east coast states
east_coast_states = [
    "ME", "NH", "VT", "MA", "RI", "CT",  # New England
    "NY", "NJ", "PA", "DE", "MD",         # Mid-Atlantic
    "VA", "NC", "SC", "GA", "FL"          # Southeast
]

energy_df = pd.concat([
    get_energy(state, "2024-01", "2024-12") for state in east_coast_states
], ignore_index=True)

# Verify it worked
print(f"Total rows: {len(energy_df)}")  # Should be 36 (12 months x 3 states)
print(f"States: {energy_df['state'].unique()}")
print(f"Date range: {energy_df['period'].min()} to {energy_df['period'].max()}")

display(energy_df)




Total rows: 192
States: ['ME' 'NH' 'VT' 'MA' 'RI' 'CT' 'NY' 'NJ' 'PA' 'DE' 'MD' 'VA' 'NC' 'SC'
 'GA' 'FL']
Date range: 2024-01 to 2024-12


,period,state,consumption_million_kwh,consumption_billion_kwh
0,2024-01,ME,471.88414,0.471884
1,2024-02,ME,474.91138,0.474911
2,2024-03,ME,435.48538,0.435485
3,2024-04,ME,431.45906,0.431459
4,2024-05,ME,390.34411,0.390344
...,...,...,...,...
187,2024-08,FL,15073.89687,15.073897
188,2024-09,FL,13738.97127,13.738971
189,2024-10,FL,11520.09567,11.520096
190,2024-11,FL,9753.14006,9.753140


In [ ]:
# Weather API location dictionary
east_coast_coords = {
    "ME": {"city": "Portland",      "lat": 43.6591, "lon": -70.2568},
    "NH": {"city": "Concord",       "lat": 43.2081, "lon": -71.5376},
    "VT": {"city": "Burlington",    "lat": 44.4759, "lon": -73.2121},
    "MA": {"city": "Boston",        "lat": 42.3601, "lon": -71.0589},
    "RI": {"city": "Providence",    "lat": 41.8240, "lon": -71.4128},
    "CT": {"city": "Hartford",      "lat": 41.7658, "lon": -72.6851},
    "NY": {"city": "New York City", "lat": 40.7128, "lon": -74.0060},
    "NJ": {"city": "Trenton",       "lat": 40.2171, "lon": -74.7429},
    "PA": {"city": "Philadelphia",  "lat": 39.9526, "lon": -75.1652},
    "DE": {"city": "Wilmington",    "lat": 39.7447, "lon": -75.5484},
    "MD": {"city": "Baltimore",     "lat": 39.2904, "lon": -76.6122},
    "VA": {"city": "Richmond",      "lat": 37.5407, "lon": -77.4360},
    "NC": {"city": "Charlotte",     "lat": 35.2271, "lon": -80.8431},
    "SC": {"city": "Columbia",      "lat": 34.0007, "lon": -81.0348},
    "GA": {"city": "Atlanta",       "lat": 33.7490, "lon": -84.3880},
    "FL": {"city": "Orlando",       "lat": 28.5383, "lon": -81.3792}
}

In [ ]:
# API call and dataframe creation for Open-Mateo Weather API

import pandas as pd

def get_weather(lat, lon, start_date, end_date, state_code, city_name):
    """
    Pulls daily weather data from Open-Meteo for a single city

    Parameters:
        lat        : float — latitude
        lon        : float — longitude
        start_date : str   — YYYY-MM-DD format
        end_date   : str   — YYYY-MM-DD format
        state_code : str   — two letter state code for labeling
        city_name  : str   — city name for labeling

    Returns:
        Pandas dataframe with daily weather rows labeled by state and city
    """
    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": [                       # daily
            "temperature_2m_max",        # daily high in fahrenheit
            "temperature_2m_min",        # daily low in fahrenheit
            "precipitation_sum",          # total daily precipitation
              "relative_humidity_2m_max",
              "relative_humidity_2m_min"
        ],
        "temperature_unit": "fahrenheit",
        "timezone": "auto"               # auto detects timezone from coordinates
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        print(f"Error {response.status_code} for {state_code} - {city_name}")
        print(response.text)
        return pd.DataFrame()

    df = pd.DataFrame(response.json()["daily"])
    df["state"] = state_code
    df["city"] = city_name
    return df

In [ ]:
# Call function for all east coast states
weather_df = pd.concat([
    get_weather(
        info["lat"],
        info["lon"],
        "2024-01-01",
        "2024-12-31",
        state,
        info["city"]
    )
    for state, info in east_coast_coords.items()
], ignore_index=True)


In [ ]:
# Show output of table to verify records
print(f"Total rows: {len(weather_df)}")
print(f"Columns: {weather_df.columns.tolist()}")
print(f"States: {weather_df['state'].unique()}")
print(f"Date range: {weather_df['time'].min()} to {weather_df['time'].max()}")

display(weather_df.head(200))

Total rows: 5856
Columns: ['time', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'relative_humidity_2m_max', 'relative_humidity_2m_min', 'state', 'city']
States: ['ME' 'NH' 'VT' 'MA' 'RI' 'CT' 'NY' 'NJ' 'PA' 'DE' 'MD' 'VA' 'NC' 'SC'
 'GA' 'FL']
Date range: 2024-01-01 to 2024-12-31


,time,temperature_2m_max,temperature_2m_min,precipitation_sum,relative_humidity_2m_max,relative_humidity_2m_min,state,city
0,2024-01-01,31.8,21.9,0.0,70,50,ME,Portland
1,2024-01-02,37.4,20.5,0.0,81,34,ME,Portland
2,2024-01-03,39.5,24.8,0.0,87,47,ME,Portland
3,2024-01-04,42.1,20.0,0.0,89,42,ME,Portland
4,2024-01-05,28.4,18.2,0.0,57,30,ME,Portland
...,...,...,...,...,...,...,...,...
195,2024-07-14,89.6,69.5,0.0,94,37,ME,Portland
196,2024-07-15,91.6,68.6,0.1,84,43,ME,Portland
197,2024-07-16,93.9,72.2,2.2,98,38,ME,Portland
198,2024-07-17,90.6,71.7,2.6,96,42,ME,Portland


In [ ]:
# Convert time (recieved as a string) column to datetime format (an actual date format) & create matching column of 'period' for tables to match

weather_df["time"] = pd.to_datetime(weather_df["time"])

weather_df["period"] = weather_df["time"].dt.to_period("M").astype(str)

# Add humidity averages before funneling into monthly averages

weather_df["humidity_avg"] = (
    weather_df["relative_humidity_2m_max"] + weather_df["relative_humidity_2m_min"]
) / 2


# Aggregate the daily weather into monthly averages

weather_monthly = weather_df.groupby(["state", "city","period"]).agg(
    avg_temp_max = ("temperature_2m_max", "mean"),
    avg_temp_min = ("temperature_2m_min", "mean"),
    avg_humidity = ("humidity_avg", "mean"),

    total_precipitation = ("precipitation_sum","sum")
).reset_index()

display(weather_monthly.head(100))

,state,city,period,avg_temp_max,avg_temp_min,avg_humidity,total_precipitation
0,CT,Hartford,2024-01,38.177419,24.580645,71.741935,169.7
1,CT,Hartford,2024-02,43.065517,24.162069,66.155172,45.3
2,CT,Hartford,2024-03,51.896774,33.538710,66.258065,207.4
3,CT,Hartford,2024-04,59.663333,39.240000,67.050000,137.9
4,CT,Hartford,2024-05,72.141935,51.809677,72.919355,104.8
...,...,...,...,...,...,...,...
95,NC,Charlotte,2024-12,52.380645,35.812903,68.532258,97.7
96,NH,Concord,2024-01,33.974194,21.106452,69.919355,156.6
97,NH,Concord,2024-02,41.493103,21.817241,60.500000,20.5
98,NH,Concord,2024-03,46.770968,28.593548,67.177419,201.3


In [ ]:
# Merge weather and energy dataframes into one

merged_data = pd.merge(
    weather_monthly,
    energy_df,
    on=["state", "period"],
    how="inner"
)

# Print the merged data
print(f"Columns:  {merged_data.columns.tolist()}")
display(merged_data.head(100))



Columns:  ['state', 'city', 'period', 'avg_temp_max', 'avg_temp_min', 'avg_humidity', 'total_precipitation', 'consumption_million_kwh', 'consumption_billion_kwh']


,state,city,period,avg_temp_max,avg_temp_min,avg_humidity,total_precipitation,consumption_million_kwh,consumption_billion_kwh
0,CT,Hartford,2024-01,38.177419,24.580645,71.741935,169.7,1274.14156,1.274142
1,CT,Hartford,2024-02,43.065517,24.162069,66.155172,45.3,1051.80176,1.051802
2,CT,Hartford,2024-03,51.896774,33.538710,66.258065,207.4,981.32315,0.981323
3,CT,Hartford,2024-04,59.663333,39.240000,67.050000,137.9,876.01541,0.876015
4,CT,Hartford,2024-05,72.141935,51.809677,72.919355,104.8,879.65058,0.879651
...,...,...,...,...,...,...,...,...,...
95,NC,Charlotte,2024-12,52.380645,35.812903,68.532258,97.7,5832.11596,5.832116
96,NH,Concord,2024-01,33.974194,21.106452,69.919355,156.6,457.53975,0.457540
97,NH,Concord,2024-02,41.493103,21.817241,60.500000,20.5,425.09955,0.425100
98,NH,Concord,2024-03,46.770968,28.593548,67.177419,201.3,389.74424,0.389744


In [ ]:
# testing and establishing connection to azure blob storage

from azure.storage.blob import BlobServiceClient
from google.colab import userdata

# Connect to azure storage account, define data archive function
def archive_to_blob(df, filename):
  conn_str = userdata.get("AZURE_CONN_STR")
  client = BlobServiceClient.from_connection_string(conn_str)
  blob = client.get_blob_client(container="raw-energy-weather-data", blob=filename)
  blob.upload_blob(df.to_csv(index=False), overwrite=True)
  print(f"Successfully uploaded {filename} to Azure Blob")

# Transform DF into CSV's and push data to Azure container
archive_to_blob(weather_df, "raw_weather_2024.csv")
archive_to_blob(energy_df, "raw_energy_2024.csv")


Successfully uploaded raw_weather_2024.csv to Azure Blob
Successfully uploaded raw_energy_2024.csv to Azure Blob


In [ ]:
# Create engine, connect to, and push data to Supabase database

from sqlalchemy import create_engine
from google.colab import userdata
from sqlalchemy import text

# Creating engine
engine = create_engine(userdata.get("SUPA_DB_KEY"))

print("engine created sucessfully")

# Testing connection of engine
with engine.connect() as connection:
  result = connection.execute(text("SELECT version()"))
  print(f"Connected successfully: {result.fetchone()[0]}")

# Creating table in supabase and pushing data frame
  merged_data.to_sql(
      "weather_energy_merged",
      engine,
      if_exists="replace",
      index=False
  )

  print(f"Succesfully written {len(merged_data)} rows to Supabase!")


engine created sucessfully
Connected successfully: PostgreSQL 17.6 on aarch64-unknown-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit
Succesfully written 192 rows to Supabase!
